# Run the OSSE with z-Only Observations

This notebook reruns the offline OSSE using the new one-channel DA configuration. Only $z$ is assimilated; the ensemble still evolves in the full three-dimensional Lorenz-63 state space. The 1012-cycle analysis history, extended to cover the selected CSE initialization at $t_0=8088$, is stored under `outputs/04_osse/z_only_obs_extended_1012/`; the original 1000-cycle history remains untouched.

The long run is intentionally identical to the validated OSSE in timing, ensemble size, model, initial-ensemble seed, and multiplicative inflation. This notebook does **not** perform an inflation sweep, because the first experiment is designed to change only the observation operator.


In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np

from class_da_sys import da_system
from class_osse import osse
from project_paths import ANALYSIS_DATA_DIR, OSSE_DATA_DIR

OBSERVATION_LABEL = 'z_only_obs'
OSSE_OUTPUT_LABEL = 'z_only_obs_extended_1012'
ANALYSIS_FILE = (
    ANALYSIS_DATA_DIR.parent / OBSERVATION_LABEL / 'data'
    / 'x_analysis_init_z_only_obs.pkl'
)
OUTPUT_ROOT = OSSE_DATA_DIR.parent / OSSE_OUTPUT_LABEL
DATA_DIR = OUTPUT_ROOT / 'data'
RESULTS_DIR = OUTPUT_ROOT / 'results'
FIGURES_DIR = OUTPUT_ROOT / 'figures'
for directory in (DATA_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

OSSE_OUTPUT_FILES = {
    'analysis_file': str(DATA_DIR / 'osse_Xa_history.pkl'),
    'prior_file': str(DATA_DIR / 'osse_Xf_history.pkl'),
    'summary_file': str(DATA_DIR / 'osse_summary.pkl'),
    'checkpoint_file': str(DATA_DIR / 'osse_Xa_history_checkpoint.npy'),
    'prior_checkpoint_file': str(DATA_DIR / 'osse_Xf_history_checkpoint.npy'),
    'checkpoint_summary_file': str(DATA_DIR / 'osse_summary_checkpoint.pkl'),
}

print('DA-system input:', ANALYSIS_FILE)
print('Isolated OSSE directory:', OUTPUT_ROOT)


## Load and Validate the z-Only DA System


In [ ]:
das = da_system().load(ANALYSIS_FILE)
sv = das.getStateVector()
x_nature = np.asarray(sv.getTrajectory(), dtype=float)
y_obs = np.asarray(das.getObsData().getVal(), dtype=float)

assert das.xdim == 3 and das.ydim == 1
assert np.array_equal(np.asarray(das.H), np.array([[0.0, 0.0, 1.0]]))
assert np.asarray(das.R).shape == (1, 1)
assert y_obs.shape == (1, x_nature.shape[1])

sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0
lorenz63_model = osse.make_lorenz63_model(sigma=sigma, rho=rho, beta=beta)

Ta = 8
t0 = 0
fcst_dt = das.fcst_dt
multiple_number = 4
T_0 = 75.1
MULTIPLICATIVE_INFLATION = 1.04

osse_exp = osse(
    das=das,
    x_nature=x_nature,
    y_obs=y_obs,
    model_func=lorenz63_model,
    sigma=sigma,
    beta=beta,
    rho=rho,
    fcst_dt=fcst_dt,
    path_figures=str(FIGURES_DIR) + '/',
)

print('H =', np.asarray(das.H))
print('Observation array shape:', y_obs.shape)
print('Nature-run shape:', x_nature.shape)


## Run the Extended Minimal OSSE

This cell runs 1012 DA cycles so the history reaches $t=8088$, and writes checkpoints plus the final analysis/prior histories to `outputs/04_osse/z_only_obs_extended_1012/`. The baseline full-observation OSSE and the earlier 1000-cycle z-only history are not touched.


In [ ]:
RUN_OSSE_FULL_NATURE_PERIOD = False
RUN_OSSE_MINIMAL_OUTPUT = True
DEFAULT_OSSE_NUM_CYCLES = 1012
OSSE_PROGRESS_EVERY = 10000
OSSE_CHECKPOINT_EVERY = 50000

osse_run = osse_exp.run_standard_osse_experiment(
    t0=t0,
    Ta=Ta,
    default_num_cycles=DEFAULT_OSSE_NUM_CYCLES,
    run_full_nature_period=RUN_OSSE_FULL_NATURE_PERIOD,
    run_minimal_output=RUN_OSSE_MINIMAL_OUTPUT,
    progress_every=OSSE_PROGRESS_EVERY,
    checkpoint_every=OSSE_CHECKPOINT_EVERY,
    multiple_number=multiple_number,
    T_0=T_0,
    multiplicative_inflation=MULTIPLICATIVE_INFLATION,
    **OSSE_OUTPUT_FILES,
    extra_summary={
        'input_das_file': str(ANALYSIS_FILE),
        'observation_mode': 'z_only',
        'observed_indices': [2],
        'observation_operator': [[0.0, 0.0, 1.0]],
        'common_random_numbers_with_full_observation_osse': True,
        'inflation_sweep_performed': False,
        'initEns_seed': 2,
    },
)

osse.publish_run_outputs(osse_run, globals(), include_full_diagnostics=False)


## Short-Window Diagnostic


In [ ]:
with open(DATA_DIR / 'osse_Xa_history.pkl', 'rb') as stream:
    saved_analysis = np.asarray(pickle.load(stream), dtype=float)
with open(DATA_DIR / 'osse_Xf_history.pkl', 'rb') as stream:
    saved_prior = np.asarray(pickle.load(stream), dtype=float)
with open(DATA_DIR / 'osse_summary.pkl', 'rb') as stream:
    saved_summary = pickle.load(stream)

assert saved_analysis.shape == saved_prior.shape
assert saved_analysis.shape[1:] == (das.xdim, das.edim)
assert saved_summary['observed_indices'] == [2]

n_plot = min(45, saved_analysis.shape[0])
times = saved_summary['t0'] + np.arange(n_plot) * saved_summary['Ta']
labels = ('x', 'y', 'z')
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

for state_index, ax in enumerate(axes):
    ax.plot(times, x_nature[state_index, times], color='black', linewidth=1.5, label='nature')
    for member in range(das.edim):
        ax.plot(times, saved_prior[:n_plot, state_index, member], color='tab:orange', alpha=0.35, linewidth=0.8)
        ax.plot(times, saved_analysis[:n_plot, state_index, member], color='tab:blue', alpha=0.45, linewidth=0.8)
    if state_index == 2:
        ax.scatter(times, y_obs[0, times], color='tab:purple', s=14, alpha=0.7, label='observed z')
    ax.set_ylabel(labels[state_index])
    ax.grid(alpha=0.2)

axes[0].plot([], [], color='tab:orange', label='prior members')
axes[0].plot([], [], color='tab:blue', label='analysis members')
axes[0].legend(frameon=False, ncol=3)
axes[2].legend(frameon=False)
axes[2].set_xlabel('Absolute time step')
fig.suptitle('OSSE analysis with z-only observations')
fig.tight_layout()
diagnostic_path = FIGURES_DIR / 'osse_z_only_obs_analysis_members.png'
fig.savefig(diagnostic_path, dpi=300, bbox_inches='tight')
plt.show()

print('Analysis history:', saved_analysis.shape)
print('Completed cycles:', saved_summary['completed_cycles'])
print('Diagnostic figure:', diagnostic_path)
